# CMAPSS full pipeline on Google Colab

**Run cells top → bottom.** One notebook does:

1. Clone this repo  
2. Install dependencies (GPU PyTorch if you use a GPU runtime)  
3. Upload NASA CMAPSS to Colab disk → import into project  
4. Phase 2 build + Phase 3 train **all four subsets (FD001–FD004)** → **MLflow logs to Databricks** (RF, GBM, LSTM, Cox PH, failure, anomaly — **no skips**)  
5. Verification report (winner + Cox NASA per subset)  
6. Zip & download models, artifacts, predictions  

Each Colab train adds **new** MLflow runs (`training_batch` tag); prior runs are **not deleted** from the experiment.

**Before you start:** `Runtime` → `Change runtime type` → **T4 GPU** (optional; LSTM only).  
**Edit cell 2:** set `REPO_URL` to your GitHub repo (HTTPS).

In [ ]:
# ── CONFIG (edit these) ──────────────────────────────────────────────
REPO_URL = "https://github.com/YOUR_USERNAME/predictive-maintenance.git"  # required
REPO_DIR = "/content/predictive-maintenance"
BRANCH = "main"  # or your feature branch

# Full training — nothing skipped (RF, GBM, LSTM, Cox, failure, anomaly)
LSTM_EPOCHS = 15
# Optional MLflow tag for this Colab session (auto timestamp if empty)
RUN_LABEL = ""  # e.g. "uc5_colab_full_v1"

# Where YOU put the 12 CMAPSS .txt files on Colab SSD (session disk)
CMAPSS_UPLOAD_DIR = "/content/cmapss_upload"

# Databricks MLflow (PAT with scope: mlflow)
DATABRICKS_HOST = "https://dbc-xxxxxxxx.cloud.databricks.com"  # your workspace base URL only
DATABRICKS_TOKEN = ""  # paste PAT here, OR use Colab Secrets (next cell)
MLFLOW_EXPERIMENT = "/Shared/predictive_maintenance"
# ─────────────────────────────────────────────────────────────────────

## 1. Clone repository

In [ ]:
import os
import shutil
from pathlib import Path

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL in the config cell to your real GitHub HTTPS URL.")

if Path(REPO_DIR).exists():
    print("Removing existing clone:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

print("Repo root:", Path.cwd())
print("Sample files:", list(Path("scripts").glob("*.py"))[:5])

## 2. Install dependencies

In [ ]:
# CUDA PyTorch for Colab GPU runtimes (safe on CPU too)
%pip install -q torch --index-url https://download.pytorch.org/whl/cu124
%pip install -q -r requirements.txt

In [ ]:
import torch
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "scripts" / "cmapss_colab_train.py").exists(), "Wrong directory — re-run clone cell"

print("Project:", ROOT)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: none (LSTM uses CPU; use TRAIN_MODE='fast' on free tier)")

## 3. Upload CMAPSS raw data (12 files)

Required files (from NASA zip `CMAPSSData/` or your PC `data/raw/cmapss/`):

`train_FD001.txt` `test_FD001.txt` `RUL_FD001.txt` … through **FD004** (12 files total).

**Option A — Colab file picker** (next cell): upload all `.txt` files.  
**Option B — Copy to SSD first:** put files in `/content/cmapss_upload/` via Colab left panel **Files → Upload**, then run **import** cell only.

In [ ]:
import shutil
from pathlib import Path

UPLOAD_DIR = Path(CMAPSS_UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("Upload folder:", UPLOAD_DIR)
print("Existing:", sorted(p.name for p in UPLOAD_DIR.glob("*.txt")))

In [ ]:
# Option A: pick files from your computer (run once; can select multiple .txt)
from google.colab import files

uploaded = files.upload()  # select all 12 train_/test_/RUL_ FD00X files
for name, data in uploaded.items():
    dest = UPLOAD_DIR / name
    dest.write_bytes(data)
    print(f"saved {name} ({len(data):,} bytes)")
print("In upload dir:", len(list(UPLOAD_DIR.glob("*.txt"))), "txt files")

In [ ]:
# Option B: unzip a NASA zip into the upload folder (skip if you uploaded .txt directly)
import zipfile

zips = list(UPLOAD_DIR.glob("*.zip")) + list(Path("/content").glob("*.zip"))
for zpath in zips:
    print("extract", zpath)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(UPLOAD_DIR)
print("txt in upload dir:", sorted(p.name for p in UPLOAD_DIR.rglob("*.txt"))[:15], "...")

In [ ]:
# Copy into project data/raw/cmapss/ (required for training)
!python scripts/import_cmapss_upload.py --source {CMAPSS_UPLOAD_DIR}

## 4. Connect MLflow → Databricks

Runs will appear in your workspace under **Machine Learning → Experiments**.

Optional: store PAT in Colab **Secrets** (key `DATABRICKS_TOKEN`) instead of pasting in config.

In [ ]:
import os

token = DATABRICKS_TOKEN
try:
    from google.colab import userdata
    if not token:
        token = userdata.get("DATABRICKS_TOKEN")
except Exception:
    pass

if not token:
    raise ValueError("Set DATABRICKS_TOKEN in config cell or Colab secret DATABRICKS_TOKEN")

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/configure_mlflow_databricks.py",
        "--host",
        DATABRICKS_HOST,
        "--token",
        token,
        "--experiment",
        MLFLOW_EXPERIMENT,
        "--smoke-test",
    ],
    cwd=ROOT,
    check=True,
)

## 5. Build features (Phase 2) + train all FD subsets (Phase 3 → Databricks MLflow)

**Full pipeline** (RF, GBM, LSTM, Cox PH, failure GBMs, Isolation Forest) for **FD001–FD004**.

**Re-runs:** each time you run this cell, MLflow creates **new** runs. Prior runs stay in the experiment (filter by tag `training_batch` or sort by start time). Only local `models/` and Parquet files are overwritten with the latest train.

Trains **FD001, FD002, FD003, FD004**. Each subset logs one parent run `FD00X_phase3_summary` with nested runs including `FD00X_rul_cox` and parent metrics `test_cox_rul_score`.

Writes:
- `data/processed/cmapss_FD00X_{train,test,predictions}.parquet` (includes `rul_pred_cox`, `survival_prob_30`)
- `artifacts/cmapss_*` + `artifacts/cmapss_training_registry.json`
- `models/rul_*`, `models/survival_FD00X.pkl`, failure/anomaly pickles
- Databricks experiment `MLFLOW_EXPERIMENT` + **Model Registry** entries (`cmapss_rul_*_FD00X`, etc.)

In [ ]:
import os

# Propagate Databricks creds to subprocess (configure cell above)
os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = token
os.environ["MLFLOW_TRACKING_URI"] = "databricks"
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT
os.environ["CMAPSS_UPLOAD_DIR"] = CMAPSS_UPLOAD_DIR

# Full train: all models, all four subsets (no --fast, no --skip-lstm, no --skip-cox)
_DATASETS = "FD001 FD002 FD003 FD004"
_run_label = f" --run-label {RUN_LABEL}" if RUN_LABEL.strip() else ""

!python scripts/cmapss_colab_train.py --lstm-epochs {LSTM_EPOCHS} --datasets {_DATASETS} --upload-dir {CMAPSS_UPLOAD_DIR} --mlflow-databricks{_run_label}

## 5b. Register models in Databricks Model Registry

Phase 3 training (cell above) **already registers** models when `MLFLOW_REGISTER_MODELS` is not `0` (default). Each subset gets names like `cmapss_rul_gbm_FD001`, `cmapss_failure_30_FD001`, `cmapss_survival_FD001`.

Run this cell only if you trained earlier without registration, or to re-register from `models/` after unzipping artifacts.

In Databricks: **Machine Learning → Models** (separate from **Experiments** run history).

In [ ]:
import os

os.environ.setdefault("MLFLOW_REGISTER_MODELS", "1")
_register_label = f" --run-label {RUN_LABEL}" if RUN_LABEL.strip() else ""

!python scripts/register_cmapss_models.py --all --mlflow-databricks{_register_label}

## 5c. Re-pull `main` + re-register (after registry fix)

**Only if the same Colab runtime still has your repo** (you did not disconnect / timeout).

- Use `git pull` — **do not** re-run the §1 clone cell (it deletes `models/`).
- In code cells, use `REPO_DIR` only — **never** put comments on the same line as `%cd`.

**If the session timed out:** skip this section. Go to **§5d After timeout** below.

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
if not repo.is_dir():
    raise FileNotFoundError(
        f"{repo} missing — Colab disk was wiped. Run §5d (restore zip) or full notebook from §1."
    )

os.chdir(repo)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "origin", "main"], check=True)

registry_src = (repo / "src/models/mlflow_registry.py").read_text()
if 'artifact_path=f"registry/' in registry_src:
    raise RuntimeError("Still on old code — pull failed or wrong branch.")
print("Repo updated:", repo)
print("Registry fix: OK")
print("models/ count:", len(list((repo / "models").glob("*"))))

In [ ]:
# Re-register all FD subsets → Databricks Model Registry (no retrain)
import os

os.environ.setdefault("MLFLOW_REGISTER_MODELS", "1")
os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = token
os.environ["MLFLOW_TRACKING_URI"] = "databricks"
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT

_register_label = " --run-label colab-register-fix"
if RUN_LABEL.strip():
    _register_label = f" --run-label {RUN_LABEL}-register"

!python scripts/register_cmapss_models.py --all --mlflow-databricks{_register_label}

In [ ]:
# Quick check: registry names in Databricks UI → Machine Learning → Models
!python scripts/report_cmapss_mlflow.py

## 5d. After Colab timeout (disk wiped)

Colab **deletes** `/content/predictive-maintenance` when the runtime ends. Your **Databricks experiment runs are safe**; local `models/` are not unless you downloaded the zip.

**Option A — You have `cmapss_colab_outputs.zip`:** upload to `/content/`, unzip into the repo, then §5c register only.

**Option B — No zip:** run §1–§4, then §5 train again (long), or register-only if you restore files manually.

In [ ]:
# Option A: upload cmapss_colab_outputs.zip via Colab Files panel to /content/, then run:
import os
import subprocess
import zipfile
from pathlib import Path

REPO_DIR = "/content/predictive-maintenance"
ZIP_PATH = Path("/content/cmapss_colab_outputs.zip")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "Upload cmapss_colab_outputs.zip to /content/ first (Files → Upload)."
    )

repo = Path(REPO_DIR)
if repo.is_dir() and (repo / ".git").exists():
    print("Repo exists — pulling latest code only")
    subprocess.run(["git", "-C", str(repo), "pull", "origin", "main"], check=True)
else:
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL in config cell, then re-run this cell.")
    subprocess.run(
        ["git", "clone", "--branch", "main", "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(REPO_DIR)

models = list(Path("models").glob("*"))
print("Restored. models/ files:", len(models))
if len(models) < 4:
    print("WARNING: few model files — zip may be incomplete; may need full §5 train.")

## 6. Supervisor verification (terminal + table)

In [ ]:
!python scripts/report_cmapss_mlflow.py

In [ ]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

registry_path = Path("artifacts/cmapss_training_registry.json")
if registry_path.exists():
    print("Registry:")
    display(pd.json_normalize(json.loads(registry_path.read_text())["datasets"]).T)

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "databricks"))
client = MlflowClient()
exp_name = os.environ.get("MLFLOW_EXPERIMENT_NAME", MLFLOW_EXPERIMENT)
exp = client.get_experiment_by_name(exp_name)
if not exp:
    print("No MLflow experiment yet.")
else:
    runs = client.search_runs([exp.experiment_id], max_results=200)
    rows = []
    for r in runs:
        name = r.info.run_name or ""
        if not name.endswith("_phase3_summary"):
            continue
        rows.append({
            "dataset": r.data.params.get("dataset_id"),
            "winner": r.data.params.get("winner"),
            "skip_lstm": r.data.params.get("skip_lstm"),
            "skip_cox": r.data.params.get("skip_cox"),
            "training_batch": r.data.params.get("training_batch"),
            "gbm_max_rows": r.data.params.get("gbm_max_train_rows"),
            "test_rmse": r.data.metrics.get("test_rmse"),
            "test_nasa": r.data.metrics.get("test_rul_score"),
            "test_cox_nasa": r.data.metrics.get("test_cox_rul_score"),
            "test_cox_rmse": r.data.metrics.get("test_cox_rmse"),
            "run_id": r.info.run_id,
        })
    display(pd.DataFrame(rows).sort_values("dataset"))

## 7. View runs in Databricks

Open your workspace → **Machine Learning → Experiments** → `predictive_maintenance` (or your `MLFLOW_EXPERIMENT` path).  
Local `mlflow ui` is only needed when `MLFLOW_TRACKING_URI=./mlruns`.

In [ ]:
print("Experiment:", os.environ.get("MLFLOW_EXPERIMENT_NAME"))
print("Workspace:", os.environ.get("DATABRICKS_HOST"))

## 8. Download artifacts to your laptop

In [ ]:
from pathlib import Path
from google.colab import files

required = ("models", "artifacts", "data/processed")
for p in required:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing {p}/ — run training cell first.")

zip_parts = "models artifacts data/processed"
if Path("mlruns").exists():
    zip_parts = "mlruns " + zip_parts
!zip -qr cmapss_colab_outputs.zip {zip_parts}
print("Zip size (MB):", round(Path("cmapss_colab_outputs.zip").stat().st_size / 1e6, 1))
files.download("cmapss_colab_outputs.zip")

## 9. On your PC (after unzip into project folder)

```bash
python scripts/report_cmapss_mlflow.py
mlflow ui --backend-store-uri ./mlruns --host 127.0.0.1 --port 5000
streamlit run dashboard/app.py
```

Dashboard sidebar lists every FD subset that has `cmapss_FD00X_predictions.parquet`.